# Notebook 03: Local-Volatility Extraction

This notebook converts the arbitrage-free option surface from Notebook 2 into a Dupire local-volatility surface. This is the most numerically delicate stage in the project because Dupire differentiates the call surface in both strike and maturity.


### Why Dupire Is Harder Than Surface Fitting
A smooth implied-volatility fit can still produce a noisy local-volatility surface if the derivatives are unstable. That is why this notebook shows both the raw Dupire output and the cleaned version. The difference between those two is not cosmetic. It is the difference between a mathematically defined object and a numerically usable one.


In [ ]:
# Cell 0: Imports
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Local imports
from local_volatility.dupire import (
    dupire_local_vol_from_call_grid,
    dupire_local_var_from_call_grid,
    DupireConfig,
)
from local_volatility.regularization import (
    regularize_local_vol,
    LocalVolRegularizationConfig,
)
from local_volatility.surface import LocalVolSurface
from utils.numerical import require_strictly_increasing

# Paths
OUTPUT_DIR = Path("../output")
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Imports complete")

In [ ]:
# Cell 1: Load Call Price Grid from Notebook 2
grid_path = OUTPUT_DIR / "call_price_grid.npz"
if not grid_path.exists():
    raise FileNotFoundError(f"Missing {grid_path}. Run Notebook 02 first.")

data = np.load(grid_path)
C_grid = data["C"]
K_grid = data["K"]
T_grid = data["T"]
S0 = float(data["S0"])
r = float(data["r"])
q = float(data["q"])

print(f"📊 Loaded call price grid: {C_grid.shape} = (nT={T_grid.size}, nK={K_grid.size})")
print(f"   S0={S0:.2f}, r={r:.4f}, q={q:.4f}")
print(f"   T: [{T_grid.min():.3f}, {T_grid.max():.3f}] years")
print(f"   K: [{K_grid.min():.1f}, {K_grid.max():.1f}]")

The call grid loaded here is already the arbitrage-free output grid from the previous notebook. That is crucial. Dupire should never be applied to a noisy or arbitrage-violating surface unless you want the resulting local vol to be dominated by numerical artifacts.


In [ ]:
# Cell 2: Extract Raw Local Volatility using Dupire Formula
cfg_dupire = DupireConfig(
    floor_density=1e-8,
    cap_sigma=5.0,
    eps_div=1e-14,
)

sigma_loc_raw = dupire_local_vol_from_call_grid(
    C=C_grid,
    K=K_grid,
    T=T_grid,
    r=r,
    q=q,
    cfg=cfg_dupire,
)

print(f"✅ Extracted raw local volatility: {sigma_loc_raw.shape}")
print(f"   Range: [{sigma_loc_raw.min():.4f}, {sigma_loc_raw.max():.4f}]")
print(f"   Mean: {sigma_loc_raw.mean():.4f}, Median: {np.median(sigma_loc_raw):.4f}")

# Check for NaN/Inf
n_nan = np.sum(np.isnan(sigma_loc_raw))
n_inf = np.sum(np.isinf(sigma_loc_raw))
if n_nan > 0 or n_inf > 0:
    print(f"⚠️  WARNING: {n_nan} NaN, {n_inf} Inf values detected!")


The raw Dupire surface often looks rough at the shortest maturities and in the far wings. This is expected: those are exactly the regions where finite differences are least stable. The notebook therefore separates extraction from regularization so the reader can see the problem before seeing the fix.


In [ ]:
# Cell 3: Regularize Local Volatility
cfg_reg = LocalVolRegularizationConfig(
    min_vol=0.05,
    max_vol=2.0,
    smooth=True,
    gaussian_sigma_T=1.0,
    gaussian_sigma_K=1.5,
    preserve_T0=True,  # Don't smooth first maturity row
)

sigma_loc = regularize_local_vol(sigma_loc_raw, cfg=cfg_reg)

print(f"✅ Regularized local volatility")
print(f"   Range: [{sigma_loc.min():.4f}, {sigma_loc.max():.4f}]")
print(f"   Mean: {sigma_loc.mean():.4f}, Median: {np.median(sigma_loc):.4f}")


Regularization in this repo is intentionally pragmatic. It repairs unstable short-dated wings first, then applies only light smoothing and clipping. The objective is not to make the surface cosmetically pretty at any cost. The objective is to obtain a surface that is cleaner while still repricing reasonably well.


In [ ]:
# Cell 4: Visualize Local Vol Surface (3D)
from mpl_toolkits.mplot3d import Axes3D

KK, TT = np.meshgrid(K_grid, T_grid)

fig = plt.figure(figsize=(14, 6))

# Raw
ax1 = fig.add_subplot(121, projection="3d")
surf1 = ax1.plot_surface(KK, TT, sigma_loc_raw, cmap="viridis", alpha=0.8)
ax1.set_xlabel("Strike K")
ax1.set_ylabel("Maturity T (years)")
ax1.set_zlabel("Local Vol (raw)")
ax1.set_title("Raw Dupire Local Volatility")
fig.colorbar(surf1, ax=ax1, shrink=0.5)

# Regularized
ax2 = fig.add_subplot(122, projection="3d")
surf2 = ax2.plot_surface(KK, TT, sigma_loc, cmap="plasma", alpha=0.8)
ax2.set_xlabel("Strike K")
ax2.set_ylabel("Maturity T (years)")
ax2.set_zlabel("Local Vol (regularized)")
ax2.set_title("Regularized Local Volatility")
fig.colorbar(surf2, ax=ax2, shrink=0.5)

plt.tight_layout()
plt.savefig(FIG_DIR / "03_local_vol_3d.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: 03_local_vol_3d.png")


In [ ]:
# Cell 5: Visualize Local Vol Slices (Fixed Maturity)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.ravel()

# Select 6 maturities
T_slices = np.linspace(T_grid.min(), T_grid.max(), 6)
for i, T_target in enumerate(T_slices):
    # Find closest T index
    idx = np.argmin(np.abs(T_grid - T_target))
    T_actual = T_grid[idx]
    
    ax = axes[i]
    ax.plot(K_grid, sigma_loc_raw[idx, :], label="Raw", alpha=0.6, linestyle="--")
    ax.plot(K_grid, sigma_loc[idx, :], label="Regularized", linewidth=2)
    ax.axvline(S0, color="red", linestyle=":", alpha=0.5, label="ATM")
    ax.set_xlabel("Strike K")
    ax.set_ylabel("Local Vol")
    ax.set_title(f"T = {T_actual:.3f} years")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "03_local_vol_slices_T.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: 03_local_vol_slices_T.png")


In [ ]:
# Cell 6: Visualize Local Vol Term Structure (Fixed Strike)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.ravel()

# Select 6 strikes (OTM put, ATM, OTM call)
K_slices = np.percentile(K_grid, [10, 30, 50, 70, 90, 95])
for i, K_target in enumerate(K_slices):
    idx = np.argmin(np.abs(K_grid - K_target))
    K_actual = K_grid[idx]
    
    ax = axes[i]
    ax.plot(T_grid, sigma_loc_raw[:, idx], label="Raw", alpha=0.6, linestyle="--")
    ax.plot(T_grid, sigma_loc[:, idx], label="Regularized", linewidth=2)
    ax.set_xlabel("Maturity T (years)")
    ax.set_ylabel("Local Vol")
    ax.set_title(f"K = {K_actual:.1f} ({100*K_actual/S0:.0f}% moneyness)")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "03_local_vol_slices_K.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: 03_local_vol_slices_K.png")


In [ ]:
# Cell 7: Heatmap Comparison (Raw vs. Regularized)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im1 = axes[0].imshow(sigma_loc_raw, aspect="auto", origin="lower", cmap="viridis", 
                     extent=[K_grid.min(), K_grid.max(), T_grid.min(), T_grid.max()])
axes[0].set_xlabel("Strike K")
axes[0].set_ylabel("Maturity T (years)")
axes[0].set_title("Raw Dupire Local Vol")
plt.colorbar(im1, ax=axes[0], label="σ_loc")

im2 = axes[1].imshow(sigma_loc, aspect="auto", origin="lower", cmap="plasma",
                     extent=[K_grid.min(), K_grid.max(), T_grid.min(), T_grid.max()])
axes[1].set_xlabel("Strike K")
axes[1].set_ylabel("Maturity T (years)")
axes[1].set_title("Regularized Local Vol")
plt.colorbar(im2, ax=axes[1], label="σ_loc")

plt.tight_layout()
plt.savefig(FIG_DIR / "03_local_vol_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: 03_local_vol_heatmap.png")


The comparison plots below should be read as diagnostics, not decoration. They show where the raw surface is unstable, where the cleaning procedure is active, and how much structure remains in the regularized surface after the worst numerical pathologies are removed.


In [ ]:
# Cell 8: Compute Local Variance and Check Stability
sigma_loc_var = sigma_loc ** 2

print("📊 Local Variance Statistics:")
print(f"   Range: [{sigma_loc_var.min():.6f}, {sigma_loc_var.max():.6f}]")
print(f"   Mean: {sigma_loc_var.mean():.6f}")

# Check for pathological gradients (sign of numerical instability)
dvar_dK = np.gradient(sigma_loc_var, axis=1)
dvar_dT = np.gradient(sigma_loc_var, axis=0)

print(f"\n📈 Gradient Magnitudes:")
print(f"   ∂(σ²)/∂K: max={np.abs(dvar_dK).max():.6f}, mean={np.abs(dvar_dK).mean():.6f}")
print(f"   ∂(σ²)/∂T: max={np.abs(dvar_dT).max():.6f}, mean={np.abs(dvar_dT).mean():.6f}")


In [ ]:
# Cell 9: Create LocalVolSurface Object
lv_surface = LocalVolSurface.from_dupire_grid(
    sigma_loc_grid=sigma_loc,
    K_grid=K_grid,
    T_grid=T_grid,
    method="linear",
    fill_value=None,  # Clamping happens inside .sigma() method
)

print(f"✅ Created LocalVolSurface")
print(f"   S_grid: [{lv_surface.S_grid.min():.1f}, {lv_surface.S_grid.max():.1f}]")
print(f"   t_grid: [{lv_surface.t_grid.min():.3f}, {lv_surface.t_grid.max():.3f}]")

# Test interpolation
S_test = np.array([S0 * 0.9, S0, S0 * 1.1])
t_test = np.array([0.25, 0.5, 1.0])
sigma_test = lv_surface.sigma(S_test, t_test)
print(f"\n🧪 Test Interpolation:")
for S, t, sig in zip(S_test, t_test, sigma_test):
    print(f"   σ_loc(S={S:.1f}, t={t:.2f}) = {sig:.4f}")


Once the local-vol grid is regularized, it is wrapped into an interpolation object. That object is what the pricing and hedging code will query later as `sigma_loc(S, t)`.


In [ ]:
# Cell 10: Save Outputs
# Save local vol grid
np.savez(
    OUTPUT_DIR / "local_vol_grid.npz",
    sigma_loc=sigma_loc,
    sigma_loc_raw=sigma_loc_raw,
    K_grid=K_grid,
    T_grid=T_grid,
    S0=S0,
    r=r,
    q=q,
)
print(f"✅ Saved: local_vol_grid.npz")

# Save LocalVolSurface for downstream notebooks
import pickle
with open(OUTPUT_DIR / "local_vol_surface.pkl", "wb") as f:
    pickle.dump(lv_surface, f)
print(f"✅ Saved: local_vol_surface.pkl")

print("\n✅ Notebook 3 complete!")
